# EduGuide AI - Personalised Learning Path Agent

**Kaggle AI Agents Capstone | Agents for Good Track**

> An AI agent that builds a free, personalised week-by-week learning roadmap for any skill or subject -- powered by the Groq API (LLaMA 3.3 70B, free).

---

### What this notebook does

1. **Installs** the `groq` Python package
2. **Configures** your free Groq API key  
3. **Runs a Python roadmap agent** that returns a structured JSON plan
4. **Runs a multi-turn chat agent** for follow-up questions
5. **Renders the full interactive UI** inline in the notebook output

---

### Agent Concepts Demonstrated

| Concept | Where |
|---|---|
| AI Agent (structured reasoning + output) | Cell 4 -- `generate_roadmap()` |
| Multi-turn conversation agent | Cell 6 -- `chat()` |
| JSON schema enforcement | Cell 4 -- system prompt |
| Deployable UI (Netlify) | Cell 7 -- HTML rendered inline |
| Security (key in config cell, not in UI source) | Cell 2 |


In [5]:
# ----------------------------------------------------------------
# CELL 1 -- Install dependencies
# ----------------------------------------------------------------
# groq is the official Python client for the Groq API.
# Groq is FREE -- get your key at https://console.groq.com

!pip install groq -q
print("groq installed")


groq installed


In [6]:
# ----------------------------------------------------------------
# CELL 2 -- Configuration
# ----------------------------------------------------------------
# Paste your FREE Groq API key below.
# Get one in 2 minutes: https://console.groq.com/keys
# Free tier gives 14,400 requests/day.

GROQ_API_KEY = "gsk_YOUR_GROQ_API_KEY_HERE"  # <-- replace this

MODEL = "llama-3.3-70b-versatile"  # fast, free, excellent for structured tasks

# Validate
if "YOUR_GROQ" in GROQ_API_KEY:
    print("Please replace GROQ_API_KEY with your key from https://console.groq.com/keys")
else:
    print(f"Key set: {GROQ_API_KEY[:12]}...{GROQ_API_KEY[-4:]}")
    print(f"Model  : {MODEL}")


Please replace GROQ_API_KEY with your key from https://console.groq.com/keys


In [7]:
# ----------------------------------------------------------------
# CELL 3 -- Groq client setup
# ----------------------------------------------------------------
import json
import re
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def call_groq(system_prompt: str, messages: list, max_tokens: int = 4096) -> str:
    """
    Core agent function.
    Sends a system prompt + conversation history to Groq LLaMA,
    returns the model reply as a plain string.

    Design:
    - Stateless: all context passed explicitly via messages list
    - Multi-turn: caller manages conversation history
    - Works for both structured (JSON) and free-form chat responses
    """
    full_messages = [{"role": "system", "content": system_prompt}] + messages

    response = client.chat.completions.create(
        model=MODEL,
        messages=full_messages,
        max_tokens=max_tokens,
        temperature=0.7,
    )
    return response.choices[0].message.content

print("Groq client ready | Model:", MODEL)


Groq client ready | Model: llama-3.3-70b-versatile


In [8]:
# ----------------------------------------------------------------
# CELL 4 -- Roadmap Generation Agent
# ----------------------------------------------------------------
# Agent design:
#   1. System prompt enforces strict JSON output schema
#   2. Single-shot LLM call generates the complete plan
#   3. JSON parse + validation ensures usable structured output
#   4. Returns a Python dict ready for display or export

ROADMAP_SYSTEM_PROMPT = """You are EduGuide AI, an expert education agent.\nYou create structured, practical, week-by-week learning roadmaps.\n\nRESPOND ONLY WITH VALID JSON -- no markdown fences, no preamble, no text outside JSON.\n\nJSON schema:\n{\n  "title": "Short roadmap title",\n  "summary": "2-sentence overview",\n  "duration": "e.g. 8 weeks",\n  "level": "beginner / intermediate / advanced",\n  "weeklyTime": "e.g. 5-7 hours/week",\n  "weeks": [\n    {\n      "week": 1,\n      "title": "Theme title",\n      "focus": "1-2 sentence theme description",\n      "topics": ["topic 1", "topic 2", "topic 3"],\n      "resources": [\n        {"name": "Resource name", "url": "https://...", "type": "video"},\n        {"name": "Resource name", "url": "https://...", "type": "course"}\n      ],\n      "milestone": "What learner can do by week end"\n    }\n  ]\n}\n\nRules:\n- 4 to 12 weeks depending on goal complexity and time available\n- 3-5 topics per week\n- 2-3 FREE resources per week (YouTube, Khan Academy, freeCodeCamp, MDN, MIT OCW, Coursera audit)\n- Milestones must be specific and motivating\n- Match difficulty to the stated experience level\n- Use real, working URLs"""

def generate_roadmap(
    goal: str,
    category: str = "General",
    level: str = "Complete beginner",
    hours_per_week: str = "5-7 hours per week",
    outcome: str = "General skill building",
    resource_types: list = None,
    extra_notes: str = ""
) -> dict:
    """
    EduGuide AI Roadmap Agent.

    Takes a student's goal and preferences, calls the Groq LLM,
    parses the structured JSON roadmap, and returns a Python dict.

    Args:
        goal:           What the student wants to learn
        category:       Subject area (e.g. 'Technology & Programming')
        level:          Experience level
        hours_per_week: Time commitment per week
        outcome:        Target outcome (job, project, exam, etc.)
        resource_types: Preferred resource formats
        extra_notes:    Special requirements or constraints

    Returns:
        dict with keys: title, summary, duration, level, weeklyTime, weeks
    """
    if resource_types is None:
        resource_types = ["YouTube videos", "Free online courses"]

    user_message = (
        f"Build a learning roadmap for:\n"
        f"Goal: {goal}\n"
        f"Category: {category}\n"
        f"Experience level: {level}\n"
        f"Time available: {hours_per_week}\n"
        f"Target outcome: {outcome}\n"
        f"Preferred resources: {', '.join(resource_types)}\n"
        f"Extra notes: {extra_notes or 'None'}"
    )

    print(f"Agent running...  Goal: {goal}")
    print(f"Level: {level} | Time: {hours_per_week}")

    raw = call_groq(ROADMAP_SYSTEM_PROMPT, [{"role": "user", "content": user_message}])

    # Strip any accidental markdown fences
    cleaned = re.sub(r'^```json\s*', '', raw.strip())
    cleaned = re.sub(r'```\s*$', '', cleaned)

    roadmap = json.loads(cleaned)

    print(f"\nRoadmap generated!")
    print(f"  Title    : {roadmap['title']}")
    print(f"  Duration : {roadmap['duration']}")
    print(f"  Weeks    : {len(roadmap['weeks'])}")
    print(f"  Level    : {roadmap['level']}")

    return roadmap


# ---- Try it -- edit these values to generate your own roadmap ----
roadmap = generate_roadmap(
    goal           = "Learn Python for data science and machine learning",
    category       = "Data Science & AI",
    level          = "Complete beginner -- no background at all",
    hours_per_week = "5-7 hours per week",
    outcome        = "Build a personal project",
    resource_types = ["YouTube videos", "Free online courses (Coursera, edX, Khan Academy)", "Practice projects"],
    extra_notes    = "Focus on practical projects, avoid heavy math theory initially"
)


Agent running...  Goal: Learn Python for data science and machine learning
Level: Complete beginner -- no background at all | Time: 5-7 hours per week


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [ ]:
# ----------------------------------------------------------------
# CELL 5 -- Display the roadmap in the notebook
# ----------------------------------------------------------------

def print_roadmap(r: dict):
    """Pretty-print a roadmap dict to the notebook output."""
    sep = "=" * 62
    print(sep)
    print(f"  {r['title']}")
    print(sep)
    print(f"  {r['summary']}")
    print(f"\n  Duration: {r['duration']}  |  Level: {r['level']}  |  {r['weeklyTime']}")
    print()

    for w in r.get("weeks", []):
        print(f"  Week {w['week']}: {w['title']}")
        print(f"  {w['focus']}")
        print("  Topics:")
        for t in w.get("topics", []):
            print(f"    - {t}")
        print("  Free Resources:")
        for res in w.get("resources", []):
            print(f"    -> {res['name']}  {res['url']}")
        if w.get("milestone"):
            print(f"  Milestone: {w['milestone']}")
        print()

print_roadmap(roadmap)


In [ ]:
# ----------------------------------------------------------------
# CELL 6 -- Multi-turn Chat Agent
# ----------------------------------------------------------------
# A separate agent that:
#   - Is aware of the student's roadmap (injected into system prompt)
#   - Maintains full conversation history across multiple calls
#   - Answers questions about learning, motivation, resources, plan
#
# Agent pattern: stateful multi-turn via explicit history list

chat_history = []  # persists across chat() calls in this session

def chat(question: str) -> str:
    """
    Multi-turn chat agent.

    Appends the question to history, calls Groq with full context,
    appends the reply, and returns it. Follow-up questions work
    naturally because the full history is included each time.

    Args:
        question: The student's question

    Returns:
        str: The agent's reply
    """
    system = (
        f"You are EduGuide AI, a warm and knowledgeable learning coach.\n"
        f"The student's roadmap: '{roadmap.get('title', '')}'. "
        f"Answer questions about their plan, learning strategies, motivation, "
        f"or resources. Be concise (2-4 sentences) and encouraging."
    )

    chat_history.append({"role": "user", "content": question})
    reply = call_groq(system, chat_history[-10:], max_tokens=512)
    chat_history.append({"role": "assistant", "content": reply})
    return reply


# ---- Demo multi-turn conversation ----
demo_questions = [
    "I feel overwhelmed. How do I stay consistent with this plan?",
    "What if I fall behind by a week -- is that okay?",
    "Which is the single most important week I should not skip?",
]

print("EduGuide AI Chat Agent -- Multi-turn Demo")
print(f"Roadmap: {roadmap.get('title', '')}\n")
print("-" * 55)

for q in demo_questions:
    print(f"\nStudent : {q}")
    reply = chat(q)
    print(f"Agent   : {reply}")


---
## Full Interactive Web UI

The cell below renders the complete EduGuide AI web app directly inside this notebook.

**Before running:** make sure `GROQ_API_KEY` in Cell 2 is set to your actual key --  
it will be automatically injected into the HTML.

> The same HTML file deploys to Netlify as a one-file static website.


In [ ]:
# ----------------------------------------------------------------
# CELL 7 -- Render the full EduGuide AI UI inside the notebook
# ----------------------------------------------------------------
# Reads index.html (upload it to your Kaggle working directory),
# injects your Groq key, and displays it as an iframe.
#
# To deploy standalone:
#   1. Save the HTML content as index.html
#   2. Go to netlify.com/drop and drag the file
#   3. Live URL in 30 seconds!

import os
from IPython.display import HTML, display

if os.path.exists("index.html"):
    with open("index.html") as f:
        html_source = f.read()

    # Auto-inject the Groq API key
    html_keyed = html_source.replace("gsk_YOUR_GROQ_API_KEY_HERE", GROQ_API_KEY)

    # Encode as data URI to avoid srcdoc escaping issues
    import base64
    encoded = base64.b64encode(html_keyed.encode()).decode()
    
    display(HTML(f'''
    <iframe src="data:text/html;base64,{encoded}"
            width="100%" height="850px"
            style="border:1px solid #1e2235; border-radius:12px;">
    </iframe>
    '''))
    print("UI rendered above!")
else:
    print("index.html not found in working directory.")
    print("Upload it alongside this notebook, then re-run this cell.")
    print("Download it from the Kaggle dataset or your GitHub repo.")


In [ ]:
# ----------------------------------------------------------------
# CELL 8 -- Save roadmap to JSON
# ----------------------------------------------------------------
# Export the generated roadmap for sharing or reuse.

import json

out = "eduguide_roadmap.json"
with open(out, "w") as f:
    json.dump(roadmap, f, indent=2)

print(f"Saved: {out}")
print(f"Weeks: {len(roadmap['weeks'])} | Duration: {roadmap['duration']}")
print()
print(json.dumps(roadmap, indent=2)[:600] + "\n  ...")


---

## Architecture

```
User Input (goal, level, time, preferences)
        |
        v
Roadmap Agent
  system prompt with JSON schema
        |
        v
  Groq API -- LLaMA 3.3 70B (free)
        |
        v
  JSON Parser + Validator
        |
        v
Interactive HTML UI (inline notebook / Netlify deploy)
        |
        v
Chat Agent (multi-turn, roadmap-aware)
  conversation history across turns
        |
        v
  Groq API -- LLaMA 3.3 70B (free)
```

### Key Design Decisions

**Why Groq?**  
Free tier, 14,400 req/day, ~1-2s inference, fully OpenAI-compatible API. Zero cost for this project.

**Why single-file HTML?**  
No build step. Works offline. Deploys to Netlify by drag-and-drop. Any student can run it.

**Why JSON schema enforcement?**  
Makes the agent output deterministic and machine-readable -- no regex scraping needed.

**Why multi-turn chat?**  
A roadmap is a starting point. Students need ongoing guidance, motivation tips, and plan adjustments.

---

## Files in this submission

| File | Purpose |
|---|---|
| `eduguide_ai.ipynb` | This notebook -- Python agent + UI demo |
| `index.html` | Standalone deployable web app |
| `README.md` | Architecture, setup, and deployment guide |
| `eduguide_roadmap.json` | Example generated roadmap (output of Cell 8) |

---

*EduGuide AI -- Kaggle AI Agents Capstone -- Agents for Good -- 2026*
